# Colab/Local vLLM QLoRA Adapter Eval

Run the prompt-engineering eval CLI with a trained QLoRA adapter from Google Colab or a local Jupyter notebook. This notebook keeps the starter notebook unchanged and uses the same evaluation path as the shell command.

In [ ]:
!nvidia-smi

## Google Drive

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except ModuleNotFoundError:
    IN_COLAB = False
    print('Not running in Colab; skipping Google Drive mount.')

## Repo Setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/DarinAnthony/151B_SP26_Competition.git'
cwd = Path.cwd().resolve()

if IN_COLAB:
    REPO_DIR = Path('/content/151B_SP26_Competition')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
elif (cwd / 'pyproject.toml').exists():
    REPO_DIR = cwd
elif (cwd.parent / 'pyproject.toml').exists():
    REPO_DIR = cwd.parent
else:
    raise RuntimeError(f'Could not find repo root from {cwd}')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo:', REPO_DIR)
print('Python:', sys.executable)

## Dependencies

Run this once if the notebook kernel does not already have the repo dependencies installed.

In [ ]:
%pip install -q "hydra-core>=1.3" peft bitsandbytes datasets accelerate transformers tqdm sympy pyyaml wandb

## Adapter and Run Config

In [ ]:
from pathlib import Path
import os

ADAPTER_PATH = Path(os.environ.get(
    'ADAPTER_PATH',
    '/content/drive/MyDrive/qwen_math_comp/outputs/qwen3_4b_numina_5k_smoketest/final_adapter' if IN_COLAB else '/cephfs/qwen_math_comp/outputs/qwen3_4b_numina_5k_smoketest/final_adapter',
))
RESULTS_DIR = Path(os.environ.get(
    'RESULTS_DIR',
    '/content/drive/MyDrive/qwen_math_comp/eval_results' if IN_COLAB else '/cephfs/qwen_math_comp/eval_results',
))
# Backup target used after each eval run. In Colab this defaults to Google Drive.
DRIVE_RESULTS_DIR = Path(os.environ.get(
    'DRIVE_RESULTS_DIR',
    '/content/drive/MyDrive/qwen_math_comp/eval_results' if IN_COLAB else str(RESULTS_DIR),
))
RUN_NAME = os.environ.get('RUN_NAME', 'sft_numina5k_sc_terse_4096_vllm_4090')
WANDB_NAME = os.environ.get('WANDB_NAME', RUN_NAME)
MAX_TOKENS = int(os.environ.get('MAX_TOKENS', '4096'))

# Used only if vLLM falls back to HF/BnB. Increase this only after the smoke
# test has enough memory headroom.
RUNNER_MICRO_BATCH_SIZE = os.environ.get('RUNNER_MICRO_BATCH_SIZE', '2')
RUNNER_PARALLEL_SAMPLES = os.environ.get('RUNNER_PARALLEL_SAMPLES', '1')

required_adapter_files = ['adapter_config.json', 'adapter_model.safetensors']
missing = [name for name in required_adapter_files if not (ADAPTER_PATH / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing adapter files under {ADAPTER_PATH}: {missing}')

os.environ['ADAPTER_PATH'] = str(ADAPTER_PATH)
os.environ['WANDB_NAME'] = WANDB_NAME
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['RUNNER_MICRO_BATCH_SIZE'] = RUNNER_MICRO_BATCH_SIZE
os.environ['RUNNER_PARALLEL_SAMPLES'] = RUNNER_PARALLEL_SAMPLES

print('ADAPTER_PATH =', os.environ['ADAPTER_PATH'])
print('RESULTS_DIR =', RESULTS_DIR)
print('DRIVE_RESULTS_DIR =', DRIVE_RESULTS_DIR)
print('RUN_NAME =', RUN_NAME)
print('WANDB_NAME =', os.environ['WANDB_NAME'])
print('MAX_TOKENS =', MAX_TOKENS)
print('RUNNER_MICRO_BATCH_SIZE =', os.environ['RUNNER_MICRO_BATCH_SIZE'])
print('RUNNER_PARALLEL_SAMPLES =', os.environ['RUNNER_PARALLEL_SAMPLES'])
print('WANDB_PROJECT =', os.environ.get('WANDB_PROJECT', '<unset; W&B disabled>'))

## Eval Helper


In [ ]:
from pathlib import Path
import shlex
import shutil
import subprocess
import sys
import time

def build_eval_cmd(run_name, *, max_tokens, results_dir, slice_indices=None):
    cmd = [
        sys.executable,
        '-m', 'experiments.prompt_engineering.src.eval',
        'run=sc_terse_only',
        f'eval.max_tokens={max_tokens}',
        'runner.engine=vllm',
        'runner.quant=bf16',
        f'runner.adapter_path={ADAPTER_PATH}',
        f'results_dir={results_dir}',
        f'run_name={run_name}',
    ]
    if slice_indices is not None:
        joined = ','.join(str(int(i)) for i in slice_indices)
        cmd.insert(4, f'eval.slice_indices=[{joined}]')
    return cmd

# Copies RESULTS_DIR/<run_name> to DRIVE_RESULTS_DIR/<run_name> after every eval.
def backup_run_to_drive(run_name, *, results_dir=RESULTS_DIR, drive_results_dir=DRIVE_RESULTS_DIR):
    src = Path(results_dir) / run_name
    dst = Path(drive_results_dir) / run_name
    if not src.exists():
        print(f'No run directory to back up yet: {src}')
        return
    if src.resolve() == dst.resolve():
        print(f'Results are already on Drive: {src}')
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Backed up results to Google Drive: {dst}')

def run_eval(run_name, *, max_tokens=MAX_TOKENS, results_dir=RESULTS_DIR, slice_indices=None, micro_batch_size=None):
    env = os.environ.copy()
    env['ADAPTER_PATH'] = str(ADAPTER_PATH)
    env['WANDB_NAME'] = WANDB_NAME
    env['TOKENIZERS_PARALLELISM'] = 'false'
    # The eval script mirrors metrics here during the run. The final backup
    # notebook cell can copy the whole run folder on demand.
    env['EVAL_METRICS_DIR'] = str(DRIVE_RESULTS_DIR)
    env['RUNNER_PARALLEL_SAMPLES'] = RUNNER_PARALLEL_SAMPLES
    env['RUNNER_MICRO_BATCH_SIZE'] = str(micro_batch_size or RUNNER_MICRO_BATCH_SIZE)

    cmd = build_eval_cmd(
        run_name,
        max_tokens=max_tokens,
        results_dir=results_dir,
        slice_indices=slice_indices,
    )

    print('ADAPTER_PATH =', env['ADAPTER_PATH'])
    print('WANDB_NAME =', env['WANDB_NAME'])
    print('DRIVE_RESULTS_DIR =', env['EVAL_METRICS_DIR'])
    print('RUNNER_MICRO_BATCH_SIZE =', env['RUNNER_MICRO_BATCH_SIZE'])
    print('RUNNER_PARALLEL_SAMPLES =', env['RUNNER_PARALLEL_SAMPLES'])
    print('Command:', ' '.join(shlex.quote(part) for part in cmd))

    start = time.time()
    subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
    print(f'Elapsed: {(time.time() - start) / 60:.1f} min')

## 5-Item Smoke Test

In [ ]:
run_eval(
    f'{RUN_NAME}_smoke5',
    max_tokens=MAX_TOKENS,
    slice_indices=[0, 1, 2, 3, 4],
)

## Full Standard Eval

In [ ]:
run_eval(
    RUN_NAME,
    max_tokens=MAX_TOKENS,
)

## Result Files

In [ ]:
from pathlib import Path

results_root = Path(RESULTS_DIR)
if not results_root.exists():
    print(f'No results directory yet: {results_root}')
else:
    for path in sorted(results_root.glob('*'))[-10:]:
        print(path)
        for metrics_name in ['metrics.json', 'leaderboard.csv', 'leaderboard.jsonl']:
            metrics_path = path / metrics_name
            if metrics_path.exists():
                print('  ', metrics_path)

## Back Up Results to Google Drive


In [ ]:
# Run this after a smoke or full eval to copy that run's output folder to Google Drive.
# The most recent smoke run is f'{RUN_NAME}_smoke5'; the full run is RUN_NAME.
backup_run_to_drive(f'{RUN_NAME}_smoke5')
backup_run_to_drive(RUN_NAME)
